In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.signal as sig
from scipy.io import wavfile
from IPython.display import Audio, display
import os
np.random.seed(0)

## Άσκηση — Ετεροσυσχέτιση και Φασματικές Πυκνότητες

Σε αυτό το εργαστήριο μελετάμε την **αυτοσυσχέτιση** και την **ετεροσυσχέτιση** σημάτων συνεχούς χρόνου, καθώς και τη σχέση τους με τις **φασματικές πυκνότητες** (ενέργειας και ισχύος) μέσω του θεωρήματος **Wiener–Khinchin**.

Όπως έχουμε αναφέρει και σε προηγούμενες ασκήσεις, ο υπολογιστής δεν γνωρίζει συνεχή χρόνο: κάθε αρχείο ήχου (WAV) είναι *ήδη δειγματοληπτημένο*, άρα μόνο μπορούμε να *προσεγγίζουμε το ολοκλήρωμα του συνεχούς χρόνου*. Θα κρατάμε συνειδητά τις μονάδες σε **δευτερόλεπτα (s)** και **Hertz (Hz)** ώστε να παραμείνουμε στη λογική του συνεχούς χρόνου — αυτή η προσέγγιση *είναι* κάτι που έχουμε κάνει πολλές φορές ως τώρα.

### 1. Σήματα ενέργειας και σήματα ισχύος

Δύο κατηγορίες σημάτων συμπεριφέρονται διαφορετικά:

* **Σήμα ενέργειας**: πεπερασμένη ενέργεια $$E_x = \int_{-\infty}^{\infty} |x(t)|^2 dt < \infty$$ (π.χ. ένας παλμός πεπερασμένης διάρκειας). Η μέση ισχύς του είναι μηδέν.

* **Σήμα ισχύος**: άπειρη ενέργεια αλλά πεπερασμένη **μέση ισχύ** $$P_x = \lim_{T\to\infty}\frac{1}{T}\int_{-T/2}^{T/2} |x(t)|^2 dt < \infty$$ (π.χ. περιοδικό ή απεριοδικό).

Αν έχετε μελετήσει τη θεωρία, καταλαβαίνετε ότι αυτή η διάκριση είναι σημαντική: καθορίζει *ποια* φασματική πυκνότητα έχει νόημα (ενέργειας ή ισχύος). 

Για παράδειγμα, η ομιλία και η μουσική μοντελοποιούνται καλύτερα ως σήματα **ισχύος**, και σύντομα θα δείτε πως χειριζόμαστε τέτοια σήματα.


**Ορισμοί συσχέτισης (πραγματικά σήματα).** Η *ετεροσυσχέτιση* δύο σημάτων ενέργειας:

$$R_{xy}(\tau) = \int_{-\infty}^{\infty} x(t)\,y(t+\tau)\,dt$$

και η *αυτοσυσχέτιση* όταν $y=x$:

$$R_{x}(\tau) = \int_{-\infty}^{\infty} x(t)\,x(t+\tau)\,dt.$$

Βασικές ιδιότητες: $R_x(\tau)=R_x(-\tau)$ (άρτια), και $R_x(0)=\int |x(t)|^2 dt = E_x$ είναι η **ενέργεια** — δηλαδή η αυτοσυσχέτιση στο $\tau=0$ μετρά «πόσο μοιάζει το σήμα με τον εαυτό του χωρίς ολίσθηση» και είναι το μέγιστο. Η μεταβλητή $\tau$ (η *υστέρηση/lag*) έχει μονάδες **χρόνου**.

### 2. Θεώρημα Wiener–Khinchin: από τη συσχέτιση στη φασματική πυκνότητα

Για ένα **σήμα ενέργειας**, η **φασματική πυκνότητα ενέργειας** (ESD) είναι

$$\Psi_x(f) = |X(f)|^2, \qquad X(f)=\int_{-\infty}^{\infty} x(t)\,e^{-j2\pi f t}\,dt,$$

και το θεώρημα **Wiener–Khinchin** λέει ότι η αυτοσυσχέτιση και η ESD αποτελούν **ζεύγος Fourier**:

$$R_x(\tau) \;\stackrel{\mathcal{F}}{\longleftrightarrow}\; \Psi_x(f),\qquad \Psi_x(f)=\int R_x(\tau)e^{-j2\pi f\tau}\,d\tau.$$

Για ένα **σήμα ισχύος** (π.χ. στάσιμο τυχαίο) ορίζουμε τη **φασματική πυκνότητα ισχύος** (PSD) $S_x(f)$ ως τον μετασχ. Fourier της αυτοσυσχέτισης ισχύος $R_x(\tau)=\lim_{T\to\infty}\frac{1}{T}\int x(t)x(t+\tau)\,dt$. Αυτή είναι η *ισχυρή* μορφή του θεωρήματος και ο μόνος σωστός τρόπος να μιλήσουμε για «φάσμα» ομιλίας ή μουσικής, που δεν έχουν πεπερασμένη ενέργεια.

Ένα διαισθητικό συμπέρασμα: **η φάση χάνεται**. Η $\Psi_x$ (ή η $S_x$) κρατά μόνο «πόση ενέργεια/ισχύς ανά συχνότητα», όχι το *πότε*. Δύο σήματα με ίδια αυτοσυσχέτιση έχουν ίδιο φάσμα ισχύος ακόμη κι αν ακούγονται τελείως διαφορετικά.

### 3. Επαλήθευση με αναλυτικό παράδειγμα (τετραγωνικός παλμός)

Πριν αγγίξουμε πραγματικό ήχο, ας **επαληθεύσουμε** την αριθμητική μας προσέγγιση εκεί όπου ξέρουμε την απάντηση με μολύβι και χαρτί. Παίρνουμε τον τετραγωνικό παλμό

$$p(t)=\begin{cases}1 & 0\le t\le T_p\\ 0 & \text{αλλού}\end{cases}$$

Ξέρουμε αναλυτικά ότι:
* Η αυτοσυσχέτιση είναι **τρίγωνο**: $R_p(\tau)=\max(T_p-|\tau|,\,0)$.
* Η ESD είναι: $\Psi_p(f)=|P(f)|^2 = T_p^2\,\mathrm{sinc}^2(fT_p)$, με $\mathrm{sinc}(u)=\dfrac{\sin(\pi u)}{\pi u}$.

Θα υπολογίσουμε και τα δύο **αριθμητικά** (ολοκλήρωμα → άθροισμα $\times\,dt$) και θα δείξουμε ότι (α) η αριθμητική αυτοσυσχέτιση πέφτει πάνω στο τρίγωνο, και (β) ο μετασχ. Fourier της αυτοσυσχέτισης ισούται με την $|P(f)|^2$ — δηλαδή το **Wiener–Khinchin επαληθεύεται αριθμητικά**.

In [ ]:
# --- Διακριτοποίηση του συνεχούς άξονα με πυκνό βήμα dt (προσέγγιση συνεχούς χρόνου) ---
fs = 20000            # πυκνή 'δειγματοληψία' για να προσεγγίσουμε το συνεχές
dt = 1 / fs
Tp = 0.01             # διάρκεια παλμού: 10 ms
t = np.arange(0, 0.03, dt)                   # άξονας χρόνου 0..30 ms
p = ((t >= 0) & (t <= Tp)).astype(float)     # τετραγωνικός παλμός

# --- Αυτοσυσχέτιση: ολοκλήρωμα -> άθροισμα * dt ---
R_num = sig.correlate(p, p, mode='full') * dt
lags = sig.correlation_lags(len(p), len(p), mode='full')
tau = lags * dt                              # υστέρηση σε ΔΕΥΤΕΡΟΛΕΠΤΑ
R_theory = np.maximum(Tp - np.abs(tau), 0.0) # αναλυτικό τρίγωνο

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(tau*1e3, R_num, lw=3, label='αριθμητικά (άθροισμα·dt)')
ax[0].plot(tau*1e3, R_theory, '--', lw=2, label=r'αναλυτικά $T_p-|\tau|$')
ax[0].set_xlim(-25, 25); ax[0].set_xlabel(r'υστέρηση $\tau$ (ms)')
ax[0].set_ylabel(r'$R_p(\tau)$'); ax[0].set_title('Αυτοσυσχέτιση τετραγωνικού παλμού')
ax[0].legend(); ax[0].grid(True)

# --- ESD: |P(f)|^2 αριθμητικά vs αναλυτικό sinc^2 vs Wiener-Khinchin ---
Nf = 4096
P = np.fft.rfft(p, Nf) * dt
f = np.fft.rfftfreq(Nf, dt)
ESD_num = np.abs(P)**2
ESD_theory = (Tp**2) * np.sinc(f*Tp)**2      # np.sinc(u)=sin(pi u)/(pi u)
Rshift = np.fft.ifftshift(R_num)             # κέντρο lag=0 στο δείγμα 0
ESD_wk = np.abs(np.fft.rfft(Rshift, Nf) * dt)
fwk = np.fft.rfftfreq(Nf, dt)

ax[1].plot(f, ESD_num, lw=3, label=r'$|P(f)|^2$ (FFT·dt)')
ax[1].plot(f, ESD_theory, '--', lw=2, label=r'$T_p^2\,\mathrm{sinc}^2(fT_p)$')
ax[1].plot(fwk, ESD_wk, ':', lw=2, label=r'$\mathcal{F}\{R_p\}$ (W–K)')
ax[1].set_xlim(0, 600); ax[1].set_xlabel('Συχνότητα (Hz)')
ax[1].set_ylabel(r'$\Psi_p(f)$'); ax[1].set_title('Φασματική πυκνότητα ενέργειας')
ax[1].legend(); ax[1].grid(True)
plt.tight_layout(); plt.show()

print('Μέγιστο σφάλμα αυτοσυσχέτισης:', np.max(np.abs(R_num - R_theory)))
print('Μέγιστο σφάλμα ESD (αριθμ. vs αναλυτικό):', np.max(np.abs(ESD_num - ESD_theory)))

Και οι τρεις καμπύλες συμπίπτουν: η αριθμητική αυτοσυσχέτιση πέφτει πάνω στο τρίγωνο, και το φάσμα που παίρνουμε απευθείας ($|P(f)|^2$) ταυτίζεται με αυτό που δίνει ο μετασχ. Fourier της αυτοσυσχέτισης. **Αυτό ακριβώς λέει το Wiener–Khinchin.** Τα μικρά υπολειμματικά σφάλματα οφείλονται αποκλειστικά στο πεπερασμένο $dt$ και $N_f$ — δηλαδή στην προσέγγιση του συνεχούς. Τώρα που εμπιστευόμαστε το εργαλείο, το εφαρμόζουμε σε πραγματικό ήχο.

### 4. Φόρτωση πραγματικών σημάτων (ομιλία & μουσική)

Τοποθετήστε τα δικά σας αρχεία ως `files/speech.wav` και `files/music.wav`. Αν λείπουν, δημιουργούνται συνθετικά σήματα ώστε το notebook να εκτελείται αυτόνομα. **Θυμηθείτε:** εδώ θεωρούμε το δειγματοληπτημένο σήμα ως *προσέγγιση* ενός σήματος συνεχούς χρόνου $x(t)$, με $T_s = 1/f_s$.

In [ ]:
# ============================================================
#  Φόρτωση σημάτων: χρησιμοποιεί τα ΔΙΚΑ ΣΑΣ αρχεία αν υπάρχουν
#  στον φάκελο  files/  (files/speech.wav, files/music.wav).
#  Αν δεν βρεθούν, δημιουργούνται ΣΥΝΘΕΤΙΚΑ σήματα ομιλίας/μουσικής
#  ώστε το notebook να τρέχει αυτόνομα για επίδειξη.
# ============================================================

def _formant(x, fc, bw, fs):
    """Απλός συντονιστής 2ης τάξης (μοντελοποιεί έναν φορμάντ)."""
    r = np.exp(-np.pi * bw / fs)
    th = 2 * np.pi * fc / fs
    a = [1.0, -2 * r * np.cos(th), r * r]
    return sig.lfilter([1.0 - r], a, x)

def synth_speech(fs=16000):
    """Συνθετική 'ομιλία': έμφωνο /a/ (περιοδικό) + άφωνο τμήμα (θόρυβος)."""
    dur_v = 0.8
    n = np.arange(int(dur_v * fs))
    f0 = 120.0
    P = int(round(fs / f0))
    e = np.zeros_like(n, dtype=float)
    e[::P] = 1.0
    v = _formant(e, 730, 90, fs)
    v = _formant(v, 1090, 110, fs)
    v = _formant(v, 2440, 160, fs)
    v /= (np.max(np.abs(v)) + 1e-12)
    sil = np.zeros(int(0.1 * fs))
    unv = np.random.randn(int(0.6 * fs))
    b, a = sig.butter(4, 3000 / (fs / 2), 'high')
    unv = sig.lfilter(b, a, unv)
    unv *= 0.3 / (np.max(np.abs(unv)) + 1e-12)
    x = np.concatenate([v, sil, unv]).astype(float)
    return fs, x / (np.max(np.abs(x)) + 1e-12)

def synth_music(fs=16000):
    """Συνθετική 'μουσική': δύο διαδοχικές συγχορδίες με αρμονικές & φθορά."""
    def chord(freqs, dur):
        n = np.arange(int(dur * fs))
        t = n / fs
        env = np.exp(-3.0 * t)
        y = np.zeros_like(t)
        for f in freqs:
            for h, amp in enumerate([1.0, 0.5, 0.33, 0.25], start=1):
                y += amp * np.sin(2 * np.pi * f * h * t)
        return y * env
    C = chord([261.63, 329.63, 392.00], 1.0)
    G = chord([392.00, 493.88, 587.33], 1.0)
    x = np.concatenate([C, G]).astype(float)
    return fs, x / (np.max(np.abs(x)) + 1e-12)

def load_signal(kind):
    """kind: 'speech' ή 'music'. Επιστρέφει (fs, x) μονοφωνικό, κανονικοποιημένο."""
    path = os.path.join('files', kind + '.wav')
    if os.path.exists(path):
        fs, x = wavfile.read(path)
        x = x.astype(np.float64)
        if x.ndim > 1:
            x = x.mean(axis=1)
        x /= (np.max(np.abs(x)) + 1e-12)
        print(f"Φορτώθηκε: {path}  (fs={fs} Hz, διάρκεια={len(x)/fs:.2f} s)")
        return fs, x
    print(f"[!] Δεν βρέθηκε {path} -> χρήση ΣΥΝΘΕΤΙΚΟΥ σήματος '{kind}'.")
    return (synth_speech() if kind == 'speech' else synth_music())

In [ ]:
fs_sp, speech = load_signal('speech')
fs_mu, music  = load_signal('music')
Ts_sp = 1 / fs_sp

t_sp = np.arange(len(speech)) / fs_sp
t_mu = np.arange(len(music)) / fs_mu
fig, ax = plt.subplots(2, 1, figsize=(12, 5))
ax[0].plot(t_sp, speech, lw=0.7); ax[0].set_title('Ομιλία x(t)')
ax[0].set_xlabel('Χρόνος (s)'); ax[0].grid(True)
ax[1].plot(t_mu, music, lw=0.7, color='C1'); ax[1].set_title('Μουσική y(t)')
ax[1].set_xlabel('Χρόνος (s)'); ax[1].grid(True)
plt.tight_layout(); plt.show()

In [ ]:
# Ακούστε τα σήματα
display(Audio(speech, rate=fs_sp))
display(Audio(music, rate=fs_mu))

### 5. Αυτοσυσχέτιση σε συνεχή χρόνο ενός έμφωνου τμήματος

Απομονώνουμε ένα σύντομο **έμφωνο** τμήμα ομιλίας (περίπου στάσιμο) και υπολογίζουμε την αυτοσυσχέτισή του ως προσέγγιση του ολοκληρώματος $R_x(\tau)=\int x(t)x(t+\tau)\,dt$. Επειδή δουλεύουμε σε συνεχή χρόνο, ο άξονας της υστέρησης $\tau$ είναι σε **χιλιοστά του δευτερολέπτου (ms)**, όχι σε δείγματα.

Η **περιοδικότητα** του έμφωνου ήχου εμφανίζεται ως κορυφές της αυτοσυσχέτισης ανά $\tau=1/f_0$: η θεμελιώδης περίοδος διαβάζεται κατευθείαν από τη θέση της πρώτης δευτερεύουσας κορυφής.

In [ ]:
# Επιλογή έμφωνου τμήματος 40 ms (έμφωνο /a/ στο συνθετικό σήμα)
seg_dur = 0.04
i0 = int(0.30 * fs_sp)
seg = speech[i0:i0 + int(seg_dur * fs_sp)]
seg = seg - np.mean(seg)                      # αφαίρεση DC

R = sig.correlate(seg, seg, mode='full') * Ts_sp     # ολοκλήρωμα -> άθροισμα*Ts
lags = sig.correlation_lags(len(seg), len(seg), mode='full')
tau = lags * Ts_sp

pos = tau > 0.002                            # αγνόησε πολύ μικρά τ
peak_idx = np.argmax(R[pos])
T0 = tau[pos][peak_idx]
print(f'Θεμελιώδης περίοδος T0 ~ {T0*1e3:.2f} ms  ->  f0 ~ {1/T0:.1f} Hz')

plt.figure(figsize=(11, 4))
plt.plot(tau*1e3, R, lw=1.2)
plt.axvline(T0*1e3, color='r', ls='--', label=f'T0 = {T0*1e3:.2f} ms (f0 = {1/T0:.0f} Hz)')
plt.xlim(-20, 20); plt.xlabel(r'υστέρηση $\tau$ (ms)'); plt.ylabel(r'$R_x(\tau)$')
plt.title('Αυτοσυσχέτιση έμφωνου τμήματος (συνεχής χρόνος)')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

### 6. Φασματική πυκνότητα ισχύος μέσω Wiener–Khinchin

Η ομιλία και η μουσική είναι σήματα **ισχύος**: ένα σκέτο $|X(f)|^2$ σε ολόκληρο το σήμα δίνει εκτιμητή με τεράστια διακύμανση («φασματικός θόρυβος»). Στο πλαίσιο του συνεχούς χρόνου, η σωστή ποσότητα είναι η PSD $S_x(f)=\mathcal{F}\{R_x(\tau)\}$. Εδώ την προσεγγίζουμε δύο τρόπους — μέσω αυτοσυσχέτισης και απευθείας — και τους συγκρίνουμε, κρατώντας τον οριζόντιο άξονα σε **Hz**.

(Στο δεύτερο, διακριτό notebook θα δούμε τη *συστηματική* λύση στη διακύμανση: τη μέθοδο **Welch**.)

In [ ]:
def psd_via_autocorr(x, fs, Nf=8192):
    x = x - np.mean(x)
    r = sig.correlate(x, x, mode='full') / len(x)   # εκτιμητής αυτοσυσχ. ισχύος
    r = np.fft.ifftshift(r)
    S = np.abs(np.fft.rfft(r, Nf)) / fs             # -> PSD (ανά Hz)
    f = np.fft.rfftfreq(Nf, 1/fs)
    return f, S

def psd_direct(x, fs, Nf=8192):
    x = x - np.mean(x)
    T = len(x) / fs
    X = np.fft.rfft(x, Nf) / fs
    f = np.fft.rfftfreq(Nf, 1/fs)
    return f, (np.abs(X)**2) / T                    # (1/T)|X(f)|^2

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
for x, fs_, name, c, a in [(speech, fs_sp, 'Ομιλία', 'C0', ax[0]),
                           (music, fs_mu, 'Μουσική', 'C1', ax[1])]:
    fa, Sa = psd_via_autocorr(x, fs_)
    fd, Sd = psd_direct(x, fs_)
    a.plot(fd, 10*np.log10(Sd + 1e-20), lw=0.5, alpha=0.5, label=r'απευθείας $\frac{1}{T}|X|^2$')
    a.plot(fa, 10*np.log10(Sa + 1e-20), lw=1.5, color=c, label=r'$\mathcal{F}\{R_x\}$ (W–K)')
    a.set_title(f'PSD — {name}'); a.set_xlim(0, 4000)
    a.set_xlabel('Συχνότητα (Hz)'); a.set_ylabel('PSD (dB/Hz)')
    a.legend(); a.grid(True)
plt.tight_layout(); plt.show()

Παρατηρήστε δύο πράγματα. Πρώτον, ο απευθείας εκτιμητής (λεπτή γραμμή) και ο μέσω αυτοσυσχέτισης εκτιμητής περιγράφουν την ίδια υποκείμενη PSD — άλλη μια επιβεβαίωση του Wiener–Khinchin. Δεύτερον, και οι δύο είναι **θορυβώδεις**: γι' αυτό χρειάζεται μέσος όρος (Welch). Στην ομιλία διακρίνονται οι φορμάντ (ευρείες κορυφές), στη μουσική οι διακριτές αρμονικές των νοτών.

### 7. Ετεροσυσχέτιση σε συνεχή χρόνο: εκτίμηση χρονικής καθυστέρησης

Η κορυφή της ετεροσυσχέτισης $R_{xy}(\tau)$ βρίσκεται στην υστέρηση όπου τα δύο σήματα «ευθυγραμμίζονται» καλύτερα — η βάση της **εκτίμησης χρονικής καθυστέρησης** (π.χ. TDOA μεταξύ δύο μικροφώνων). Φτιάχνουμε μια **κλασματικά** καθυστερημένη εκδοχή $y(t)=x(t-\tau_0)$ (η κλασματική καθυστέρηση τονίζει ότι είμαστε σε συνεχή χρόνο) και ανακτούμε το $\tau_0$ **σε δευτερόλεπτα** από τη θέση της κορυφής.

In [ ]:
tau0 = 0.0037                                # πραγματική καθυστέρηση: 3.7 ms
x = speech[int(0.25*fs_sp):int(0.65*fs_sp)].copy()
x = x - np.mean(x)

# Κλασματική καθυστέρηση μέσω γραμμικής φάσης στη συχνότητα
N = len(x)
freqs = np.fft.rfftfreq(N, Ts_sp)
X = np.fft.rfft(x)
y = np.fft.irfft(X * np.exp(-1j*2*np.pi*freqs*tau0), n=N)
y = y + 0.05*np.random.randn(N)              # λίγος θόρυβος για ρεαλισμό

Rxy = sig.correlate(y, x, mode='full') * Ts_sp
lags = sig.correlation_lags(len(y), len(x), mode='full')
tau = lags * Ts_sp
tau_est = tau[np.argmax(Rxy)]
print(f'Πραγματική καθυστέρηση : {tau0*1e3:.2f} ms')
print(f'Εκτιμώμενη καθυστέρηση : {tau_est*1e3:.2f} ms')

plt.figure(figsize=(11, 4))
plt.plot(tau*1e3, Rxy, lw=1.0)
plt.axvline(tau_est*1e3, color='r', ls='--', label=f'κορυφή στα {tau_est*1e3:.2f} ms')
plt.xlim(-20, 20); plt.xlabel(r'υστέρηση $\tau$ (ms)'); plt.ylabel(r'$R_{xy}(\tau)$')
plt.title('Ετεροσυσχέτιση: εκτίμηση χρονικής καθυστέρησης')
plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

Η κορυφή εντοπίζει σωστά την καθυστέρηση, με ακρίβεια ενός δείγματος ($T_s$). Για υπο-δειγματική ακρίβεια θα χρειαζόταν παρεμβολή γύρω από την κορυφή — κάτι φυσικό στο συνεχές πλαίσιο.

### Σύνοψη

Είδαμε τους ορισμούς συνεχούς χρόνου (αυτο/ετεροσυσχέτιση, ESD/PSD), επαληθεύσαμε αριθμητικά το θεώρημα **Wiener–Khinchin** σε αναλυτικό παράδειγμα, και το εφαρμόσαμε σε ομιλία/μουσική για εκτίμηση θεμελιώδους συχνότητας, φάσματος ισχύος και χρονικής καθυστέρησης — κρατώντας παντού μονάδες **s** και **Hz**. Σε όλη τη διαδρομή, κάθε ολοκλήρωμα ήταν στην πράξη ένα άθροισμα $\times\,T_s$: μια *προσέγγιση* του συνεχούς.

---
---